In [ ]:
import json
import os
from glob import glob
import pandas as pd
from IPython.display import display, Markdown

# Find the latest experiment summary JSON
json_files = sorted(glob("../experiment_results/experiment_summary_*.json"))
latest_file = json_files[-1]
print(f"Using: {latest_file}")

with open(latest_file) as f:
    data = json.load(f)

# Mapping: internal method name -> display name
method_map = {
    "dist_spec": "DSD",
    "dist_split_spec": "DSSD",
    "uncertainty_decoding": "CUHLM",
    "adaptive_tridecoding": r"CEE-SD",
}

# Mapping: internal model name -> display name
model_map = {
    "llama-2-13b": "Llama",
    "Qwen/Qwen1.5-7B-Chat": "Qwen1.5",
    "Qwen/Qwen3-14B": "Qwen3",
}

# Mapping: dataset path fragment -> display name
dataset_map = {
    "mt_bench": "MTBench",
    "gsm8k": "GSM8K",
    "humaneval": "HumanEval",
}

method_order = ["DSD", "DSSD", "CUHLM", r"CEE-SD"]
dataset_order = ["MTBench", "GSM8K", "HumanEval"]


def get_dataset_name(dataset_path):
    for key, name in dataset_map.items():
        if key in dataset_path.lower():
            return name
    return dataset_path


# Build lookup: (model, method, dataset) -> dict of metrics
lookup = {}
for item in data:
    if item.get("status") != "success":
        continue
    cfg = item["config"]
    res = item["result"]
    mode = cfg.get("eval_mode", "")
    if mode not in method_map:
        continue
    method = method_map[mode]
    model = model_map.get(cfg.get("target_model", ""), "")
    if not model:
        continue
    dataset = get_dataset_name(cfg.get("eval_dataset", ""))
    if not dataset:
        continue

    gen_tokens = res.get("generated_tokens", 0)
    if gen_tokens <= 0:
        continue

    # Thr: end-to-end token throughput
    thr = res.get("throughput", 0)

    # C_comm: WAN communication data per token (bytes)
    edge_cloud_bytes = res.get("edge_cloud_data_bytes", 0)
    ec_mb = round(edge_cloud_bytes / (1024 * 1024), 2)
    c_comm = round(ec_mb * 1024 * 1024 / gen_tokens, 2)

    # R_acce: acceptance rate of cloud LLM (%)
    draft_acc = res.get("draft_accepted_tokens", 0)
    draft_gen = res.get("draft_generated_tokens", 0)
    if mode == "uncertainty_decoding":
        r_acce = None
    elif draft_gen > 0:
        r_acce = round(draft_acc / draft_gen * 100, 2)
    else:
        r_acce = None

    # T_comm: communication time per token (ms)
    comm_time = res.get("communication_time", 0)
    t_comm = round(comm_time / gen_tokens * 1000, 1)

    # Fwd: forward times of cloud LLM
    fwd = int(res.get("target_forward_times", 0))

    # Cost: cost of inference
    draft_wall_time = res.get("draft_wall_time", 0.0)
    target_wall_time = res.get("target_wall_time", 0.0)
    cost = (
        4.4 * res.get("wall_time") / 3600
        if cfg.get("method") == "adaptive_tridecoding"
        or ("CEE" in method)
        or ("cee" in method)
        else 4.05 * res.get("wall_time") / 3600
    )

    lookup[(model, method, dataset)] = {
        "thr": thr,
        "c_comm": c_comm,
        "r_acce": r_acce,
        "t_comm": t_comm,
        "fwd": fwd,
        "cost": cost,
        # Per-model computation time (new fields)
        "comp_little": res.get("little_computation_time", 0),
        "comp_draft": res.get("draft_computation_time", 0),
        "comp_target": res.get("target_computation_time", 0),
    }

# --- Table 1: Main metrics ---
rows = []
for model_display in ["Llama", "Qwen1.5", "Qwen3"]:
    for method in method_order:
        row = {"Model": model_display, "Method": method}
        for dataset in dataset_order:
            key = (model_display, method, dataset)
            d = lookup.get(key)
            if d:
                row[f"{dataset}_Thr"] = f"{d['thr']:.2f}"
                row[f"{dataset}_Ccomm"] = f"{d['c_comm']:.2f}"
                row[f"{dataset}_Racce"] = (
                    f"{d['r_acce']:.2f}" if d["r_acce"] is not None else "-"
                )
                row[f"{dataset}_Tcomm"] = f"{d['t_comm']:.2f}"
                row[f"{dataset}_Fwd"] = f"{d['fwd']}"
                row[f"{dataset}_Cost"] = f"{d['cost']}"
            else:
                for suffix in ["Thr", "Ccomm", "Racce", "Tcomm", "Fwd", "Cost"]:
                    row[f"{dataset}_{suffix}"] = "N/A"
        rows.append(row)

cols = pd.MultiIndex.from_tuples(
    [
        ("", "Model"),
        ("", "Method"),
        ("MTBench", "Thr."),
        ("MTBench", "$C_{comm}$"),
        ("MTBench", "$R_{acce}$"),
        ("MTBench", "$T_{comm}$"),
        ("MTBench", "Fwd."),
        ("MTBench", "cost"),
        ("GSM8K", "Thr."),
        ("GSM8K", "$C_{comm}$"),
        ("GSM8K", "$R_{acce}$"),
        ("GSM8K", "$T_{comm}$"),
        ("GSM8K", "Fwd."),
        ("GSM8K", "cost"),
        ("HumanEval", "Thr."),
        ("HumanEval", "$C_{comm}$"),
        ("HumanEval", "$R_{acce}$"),
        ("HumanEval", "$T_{comm}$"),
        ("HumanEval", "Fwd."),
        ("HumanEval", "cost"),
    ]
)

vals = []
for row in rows:
    vals.append(
        [
            row["Model"],
            row["Method"],
            row["MTBench_Thr"],
            row["MTBench_Ccomm"],
            row["MTBench_Racce"],
            row["MTBench_Tcomm"],
            row["MTBench_Fwd"],
            row["MTBench_Cost"],
            row["GSM8K_Thr"],
            row["GSM8K_Ccomm"],
            row["GSM8K_Racce"],
            row["GSM8K_Tcomm"],
            row["GSM8K_Fwd"],
            row["GSM8K_Cost"],
            row["HumanEval_Thr"],
            row["HumanEval_Ccomm"],
            row["HumanEval_Racce"],
            row["HumanEval_Tcomm"],
            row["HumanEval_Fwd"],
            row["HumanEval_Cost"],
        ]
    )

df = pd.DataFrame(vals, columns=cols)
display(Markdown("# Main Experiment Results"))
display(df)

# --- Table 2: Per-Model Computation Time ---
timing_rows = []
for model_display in ["Llama", "Qwen1.5", "Qwen3"]:
    for method in method_order:
        row = {"Model": model_display, "Method": method}
        for dataset in dataset_order:
            key = (model_display, method, dataset)
            d = lookup.get(key, {})
            if d:
                row[f"{dataset}_CompL"] = f"{d['comp_little']:.2f}"
                row[f"{dataset}_CompD"] = f"{d['comp_draft']:.2f}"
                row[f"{dataset}_CompT"] = f"{d['comp_target']:.2f}"
            else:
                for s in ["CompL", "CompD", "CompT"]:
                    row[f"{dataset}_{s}"] = "N/A"
        timing_rows.append(row)

timing_cols = pd.MultiIndex.from_tuples(
    [
        ("", "Model"),
        ("", "Method"),
        ("MTBench", "L.Comp(s)"),
        ("MTBench", "D.Comp(s)"),
        ("MTBench", "T.Comp(s)"),
        ("GSM8K", "L.Comp(s)"),
        ("GSM8K", "D.Comp(s)"),
        ("GSM8K", "T.Comp(s)"),
        ("HumanEval", "L.Comp(s)"),
        ("HumanEval", "D.Comp(s)"),
        ("HumanEval", "T.Comp(s)"),
    ]
)

timing_vals = []
for row in timing_rows:
    timing_vals.append(
        [
            row["Model"],
            row["Method"],
            row["MTBench_CompL"],
            row["MTBench_CompD"],
            row["MTBench_CompT"],
            row["GSM8K_CompL"],
            row["GSM8K_CompD"],
            row["GSM8K_CompT"],
            row["HumanEval_CompL"],
            row["HumanEval_CompD"],
            row["HumanEval_CompT"],
        ]
    )

timing_df = pd.DataFrame(timing_vals, columns=timing_cols)
display(
    Markdown(
        "## Per-Model Computation Time (s)\n*L=Little, D=Draft, T=Target. Requires re-running experiments.*"
    )
)
display(timing_df)

# Print LaTeX table rows
print("\n" + "=" * 80)
for row in rows:
    model = row["Model"]
    method = row["Method"]
    parts = [model, method]
    for ds in ["MTBench", "GSM8K", "HumanEval"]:
        parts.append(row[f"{ds}_Thr"])
        parts.append(row[f"{ds}_Ccomm"])
        parts.append(row[f"{ds}_Racce"])
        parts.append(row[f"{ds}_Tcomm"])
        parts.append(row[f"{ds}_Fwd"])
        parts.append(row[f"{ds}_Cost"])
    print(" & ".join(parts) + " \\\\")